# 03 — cosine similarity


In [86]:
import os
import sys
import shutil
import pyspark
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import BucketedRandomProjectionLSH
import pandas as pd
import numpy as np
from pyspark.sql.functions import col
from sklearn.metrics.pairwise import cosine_similarity

## THE NEXT TWO CELLS ARE CRUCIAL FOR RUNNING HADOOP AND PYSPARK ON MY DEVICE


In [87]:
import sys

# On Windows, the system-level SPARK_HOME may point to a different Spark version.
# We override it here to make sure PySpark always finds its own bundled JARs.
os.environ["SPARK_HOME"]  = os.path.dirname(pyspark.__file__)

# Spark needs Hadoop's winutils.exe on Windows to handle local file operations
# (directory creation, temp files, etc.). We installed winutils at C:\hadoop\bin\.
os.environ["HADOOP_HOME"] = r"C:\hadoop"

# Prepend C:\hadoop\bin to PATH so the JVM can find hadoop.dll via
# System.loadLibrary("hadoop"). HADOOP_HOME alone only locates winutils.exe
# for Hadoop's shell helpers — it does NOT add the directory to
# java.library.path. Without this, NativeIO$Windows.access0 throws
# UnsatisfiedLinkError on Spark write operations.
hadoop_bin = r"C:\hadoop\bin"
if hadoop_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = hadoop_bin + os.pathsep + os.environ.get("PATH", "")

# Pin the Python interpreter for both the driver and the executor-side workers
# to the exact Python running this notebook. Without these, PySpark on Windows
# can pick a different Python for workers, causing them to die with
# "Connection reset" the moment Spark tries to deserialize Python rows.
os.environ["PYSPARK_PYTHON"]        = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("SPARK_HOME    :", os.environ["SPARK_HOME"])
print("HADOOP_HOME   :", os.environ["HADOOP_HOME"])
print("PATH[0]       :", os.environ["PATH"].split(os.pathsep)[0])
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])

SPARK_HOME    : c:\Users\Mariam\anaconda3\envs\spotify-recommender\Lib\site-packages\pyspark
HADOOP_HOME   : C:\hadoop
PATH[0]       : C:\hadoop\bin
PYSPARK_PYTHON: c:\Users\Mariam\anaconda3\envs\spotify-recommender\python.exe


In [88]:
# local[*]  — uses all CPU cores on this machine (pseudo-distributed mode).
# spark.driver.memory — heap given to the JVM driver process.
# spark.local.dir     — MUST be on the same drive as the output directory (D:).
#   Spark writes temp files here, then renames them to the final output path.
#   Java's File.renameTo() cannot rename across different drives on Windows,
#   so if temp is on C: and output is on D:, every write fails silently.
# spark.hadoop.home.dir — tells the JVM where winutils lives at runtime.
# RawLocalFileSystem  — skips Unix chmod/chown calls that crash on Windows.
# fileoutputcommitter.algorithm.version=2 — avoids the rename-based commit phase.
# marksuccessfuljobs=false — skips writing the _SUCCESS marker file.

SPARK_TEMP_DIR = "D:/tmp/spark"
os.makedirs(SPARK_TEMP_DIR, exist_ok=True)

spark = SparkSession.builder \
    .appName("SpotifyPreprocessing") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.local.dir", SPARK_TEMP_DIR) \
    .config("spark.hadoop.home.dir", r"C:\hadoop") \
    .config("spark.hadoop.fs.file.impl", "org.apache.hadoop.fs.RawLocalFileSystem") \
    .config("spark.hadoop.fs.file.impl.disable.cache", "true") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.marksuccessfuljobs", "false") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)


def spark_write_csv(df, relative_path):
    """
    Write a Spark DataFrame to a single CSV file on Windows.
    Deletes the output directory with Python first so Spark never has to
    rename or overwrite an existing folder across drives.
    """
    if os.path.exists(relative_path):
        shutil.rmtree(relative_path)
    abs_path = os.path.abspath(relative_path).replace("\\", "/")
    df.coalesce(1).write.csv(f"file:///{abs_path}", header=True)
    print(f"Saved → {relative_path}/")

Spark version: 3.5.8


In [89]:
songs_df = spark.read.csv(
    "../data/processed/songs_cleaned",
    header=True,
    inferSchema=True,
)
print(f"Loaded {songs_df.count():,} songs")
songs_df.printSchema()
songs_df.show(5, truncate=False)

Loaded 738,904 songs
root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- duration: double (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)

+----------------------+-------------------------------------------------------------------+-----------------+-------------------+-------------------+------+--------------------+---------------------+----------------+--------+------------------+------------------+
|track_id              |track_name                                                         |artist_name      |duration           |danceability       |energy|speechiness         |acousticness         |instrument

In [90]:
AUDIO_FEATURES = [
    "duration",
    "danceability", "energy", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence", "tempo"
]
assembler = VectorAssembler(inputCols=AUDIO_FEATURES, outputCol="features")
songs_df = assembler.transform(songs_df).drop(*AUDIO_FEATURES)
print("After assembling features:")
songs_df.show(5, truncate=False)

After assembling features:
+----------------------+-------------------------------------------------------------------+-----------------+----------------------------------------------------------------------------------------------------------------------------------------+
|track_id              |track_name                                                         |artist_name      |features                                                                                                                                |
+----------------------+-------------------------------------------------------------------+-----------------+----------------------------------------------------------------------------------------------------------------------------------------+
|0wtpkz93wATDkUExJVuXEl|""" la traviata "" : amami alfredo (act ii) - digitally remastered"|maria callas     |[0.1171837637494022,0.36436436436436437,0.275,0.0443298969072165,0.996987951807229,0.0284,0.293,0.0394,0.34440741970453

In [91]:
lsh = BucketedRandomProjectionLSH(
    inputCol="features",
    outputCol="hashes",
    bucketLength=2.0,
    numHashTables=3,
)
lsh_model = lsh.fit(songs_df)
print("LSH model fitted to the songs dataset.")

LSH model fitted to the songs dataset.


In [92]:
playlist_pdf = pd.read_csv("../outputs/playlists/old_playlist.csv")

# normalize to match songs_cleaned format (lowercased + trimmed)
playlist_pdf["track_name"]  = playlist_pdf["track_name"].str.lower().str.strip()
playlist_pdf["artist_name"] = playlist_pdf["artist_name"].str.lower().str.strip()

has_track_id = (
    "track_id" in playlist_pdf.columns
    and playlist_pdf["track_id"].notna().any()
)
if has_track_id:
    playlist_pdf["track_id"] = playlist_pdf["track_id"].fillna("").str.strip()

print(f"Playlist loaded: {len(playlist_pdf)} songs")
playlist_pdf.head()

Playlist loaded: 1 songs


,track_id,track_name,artist_name
0,0Ip3S2GxWd2xMA360FZB3F,غير كل البنات,dolly shahine


In [93]:
playlist_spark = spark.createDataFrame(playlist_pdf)

if has_track_id:
    with_id    = playlist_spark.filter(col("track_id") != "")
    without_id = playlist_spark.filter(col("track_id") == "")

    matched_by_id   = songs_df.join(with_id.select("track_id"), on="track_id", how="inner")
    matched_by_name = songs_df.join(without_id.select("track_name", "artist_name"), on=["track_name", "artist_name"], how="inner")
else:
    matched_by_id   = songs_df.filter("false")  # empty
    matched_by_name = songs_df.join(playlist_spark.select("track_name", "artist_name"), on=["track_name", "artist_name"], how="inner")

matched_df = (
    matched_by_id
    .unionByName(matched_by_name)
    .dropDuplicates(["track_id"])
)

print(f"Matched {matched_df.count()} songs (by id + name)")
matched_df.show(5, truncate=False)

Matched 1 songs (by id + name)
+----------------------+-------------+-------------+---------------------------------------------------------------------------------------------------------------------------+
|track_id              |track_name   |artist_name  |features                                                                                                                   |
+----------------------+-------------+-------------+---------------------------------------------------------------------------------------------------------------------------+
|0Ip3S2GxWd2xMA360FZB3F|غير كل البنات|dolly shahine|[0.178697393591583,0.43043043043043044,0.747,0.06030927835051547,0.2520080321285141,2.77E-5,0.175,0.442,0.7587995983726894]|
+----------------------+-------------+-------------+---------------------------------------------------------------------------------------------------------------------------+



In [94]:
matched_rows = matched_df.collect()

# only exclude the original playlist songs, not all artist-matched songs
original_playlist_ids = set(playlist_pdf["track_id"].tolist()) if has_track_id else set()

# --- LSH candidates ---
all_candidates = None
for row in matched_rows:
    candidates = lsh_model.approxNearestNeighbors(
        songs_df,
        row["features"],
        numNearestNeighbors=50,
    ).drop("hashes", "distCol")
    all_candidates = candidates if all_candidates is None else all_candidates.unionByName(candidates)

all_candidates = (
    all_candidates
    .dropDuplicates(["track_id"])
    .filter(~col("track_id").isin(original_playlist_ids))
)

# --- Inject same-artist songs directly into the pool ---
playlist_artists_list = playlist_pdf["artist_name"].str.lower().str.strip().tolist()
same_artist_songs = (
    songs_df
    .filter(col("artist_name").isin(playlist_artists_list))
    .filter(~col("track_id").isin(original_playlist_ids))
)

same_artist_count = same_artist_songs.count()
print(f"Same-artist songs found in dataset (excluding playlist): {same_artist_count:,}")
if same_artist_count > 0:
    same_artist_songs.select("track_name", "artist_name").show(10, truncate=False)
else:
    print("⚠ None of the playlist artists have other songs in songs_cleaned.")
    print("Playlist artists searched for:")
    for a in sorted(set(playlist_artists_list)):
        print(f"  - {a}")

all_candidates = (
    all_candidates
    .unionByName(same_artist_songs)
    .dropDuplicates(["track_id"])
)
print(f"\nTotal candidates after dedup: {all_candidates.count():,}")

Same-artist songs found in dataset (excluding playlist): 32
+-------------------+-------------+
|track_name         |artist_name  |
+-------------------+-------------+
|ana a'er kol elbnat|dolly shahine|
|atnett             |dolly shahine|
|راح اللى راح       |dolly shahine|
|ana zay ay bent    |dolly shahine|
|moumou eini        |dolly shahine|
|جديد عليا          |dolly shahine|
|زى القمر           |dolly shahine|
|je suis malade     |dolly shahine|
|اتنطط              |dolly shahine|
|ولا كل البنات      |dolly shahine|
+-------------------+-------------+
only showing top 10 rows


Total candidates after dedup: 81


In [95]:
playlist_size = 50 # Adjust this based on how many songs are in the playlist
artist_slots = 10

In [96]:
candidates_pdf = all_candidates.toPandas()

playlist_vecs  = np.array([row["features"].toArray() for row in matched_rows])
candidate_vecs = np.vstack(candidates_pdf["features"].apply(lambda v: v.toArray()))

sim_matrix = cosine_similarity(candidate_vecs, playlist_vecs)
candidates_pdf["mean_cosine_score"] = sim_matrix.mean(axis=1)

# --- Top 20 overall by cosine score ---
top_overall = (
    candidates_pdf
    .sort_values("mean_cosine_score", ascending=False)
    .head(playlist_size - artist_slots)
)

# --- Top 10 same-artist slots ---
playlist_artists = set(playlist_pdf["artist_name"].str.lower().str.strip().tolist())
same_artist_df   = candidates_pdf[candidates_pdf["artist_name"].isin(playlist_artists)]

# exclude artists already in the top 20
already_in = set(top_overall.index)
same_artist_df = same_artist_df[~same_artist_df.index.isin(already_in)]

top_artist = (
    same_artist_df
    .sort_values("mean_cosine_score", ascending=False)
    .head(artist_slots)
)

# --- Combine and display ---
top = (
    pd.concat([top_overall, top_artist])
    .drop_duplicates(subset=["track_name", "artist_name"])
    .reset_index(drop=True)[["track_name", "artist_name", "mean_cosine_score"]]
)

print(f"Top {playlist_size} recommendations ({len(top_overall)} overall + {len(top_artist)} same-artist):")
display(top)

os.makedirs("../outputs/playlists", exist_ok=True)
top.to_csv("../outputs/playlists/suggestions.csv", index=False)
print("Saved → ../outputs/playlists/suggestions.csv")

Top 50 recommendations (40 overall + 10 same-artist):


,track_name,artist_name,mean_cosine_score
0,finally healing,abandoned;shiah maisel,0.998710
1,like we used to,a rocket to the moon,0.997902
2,make me feel weeeird,together pangea,0.997645
3,recuerdame bonito,miguel angel y su grupo carino,0.997069
4,on & on,cartoon;daniel levi,0.996925
5,one chance,modest mouse,0.996847
6,it just comes natural (full vocal version) [in...,karaoke library,0.996833
7,mud on the tires,brad paisley,0.996829
8,leaving new orleans,jordan davis,0.996614
9,christmas on george street,jeremy goulding,0.996599


Saved → ../outputs/playlists/suggestions.csv


In [97]:
spark.stop()
print("Done. Spark session closed.")

Done. Spark session closed.
